## 最小限の語彙の追加

最小限の語彙の追加で、コーパス内で未知語がでないようにする操作

In [1]:
# 語彙を追加する先のモデル
BASE_BERT_MODEL = "alabnii/jmedroberta-base-sentencepiece"
CHAR_ADD_MODEL_PATH = "jmedroberta_bf_randinit"

# 使用するコーパスのリスト
CORPUS_LIST = [
    "../../resources/corpus/chui_word_dic_corpus.txt",
    "../../resources/corpus/sho_db_corpus.txt",
    "../../resources/corpus/who_tcm_corpus.txt"
]

corpus_text_list = []


for corpus_path in  CORPUS_LIST:
    with open(corpus_path, mode="r", encoding="utf-8") as fp:
        lines = fp.readlines()
        corpus_text_list += lines


corpus_text_list = [
    text.replace("\n", "") for text in corpus_text_list if text != "\n" or text != ""
]

import unicodedata

normalized_text_list = [ unicodedata.normalize("NFKC", text) for text in corpus_text_list]

# 文字単位で最小限の追加
char_dict = {}

for text in normalized_text_list:
    text = text.replace("\n", "")
    if text == "\n":
        continue
    for char in text:
        if char == "":
            continue
        if char not in char_dict:
            char_dict[char] = 0
        char_dict[char] += 1


print(len(char_dict))
print(char_dict)

char_list = [char for char in char_dict.keys()]

from transformers import AutoTokenizer, AutoModel

temp_tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(BASE_BERT_MODEL)

base_token_max_id = len(temp_tokenizer)
print(base_token_max_id)

unk_char_list = []
empty_char_count = 0

for char in char_list:
    token = temp_tokenizer.tokenize(char)
    if len(token) == 0:
        empty_char_count += 1
        continue
    if token[0] == "[UNK]":
        unk_char_list.append(char)

print(empty_char_count)

temp_tokenizer.add_tokens(unk_char_list, special_tokens=False)

unk_count = 0
unk_tokenize_word_set = set()


# 文字単位で追加しても未知語になった個所の追加
for text in corpus_text_list:
    norm_text = unicodedata.normalize('NFKC', text.strip("\n").strip())
    ids = temp_tokenizer.encode(norm_text, return_tensors='pt')
    tokens = temp_tokenizer.convert_ids_to_tokens(ids[0])
    if ("[UNK]" in tokens):
        new_text = norm_text
        for i in range(0, len(tokens)):
            tokens[i] = tokens[i].removeprefix("##")

        for i in range(0, len(tokens)):
            if (tokens[i] == "[UNK]"):
                next_known_token = ""
                for j in  range(i + 1, len(tokens)):
                    if (tokens[j] != "[UNK]"):
                        next_known_token = tokens[j]
                        break
                if next_known_token != "":
                    next_token_start_index = \
                        new_text.find(next_known_token)
                    new_word = new_text[:next_token_start_index]
                    if new_word != "":
                        unk_tokenize_word_set.add(new_word)
                    new_text = new_text[next_token_start_index:]
                else:
                    if new_text != "":
                        unk_tokenize_word_set.add(new_text)
                    break
            else:
                new_text1 = new_text.removeprefix(tokens[i])
                new_text2 = new_text.removeprefix(tokens[i] + " ")
                new_text = new_text1 \
                    if (len(new_text1) < len(new_text2)) else new_text2 


print(len(unk_tokenize_word_set))

print(unk_tokenize_word_set)

temp_tokenizer.add_tokens(list(unk_tokenize_word_set), special_tokens=False)



for text in corpus_text_list:
    tokens = temp_tokenizer.tokenize(text)
    if "[UNK]" in tokens:
        print("------")
        print(text)
        print(tokens)


max_id = len(temp_tokenizer)
print(max_id)
used_ids = set()
unuse_ids = set()
for text in corpus_text_list:
    ids = temp_tokenizer.encode(text)
    for id in ids:
        used_ids.add(id)

for i in range(base_token_max_id, max_id):
    if i not in used_ids:
        unuse_ids.add(i)

print(list(unuse_ids))

from transformers import BertForPreTraining

temp_tokenizer.save_pretrained(CHAR_ADD_MODEL_PATH)
bert_model: BertForPreTraining = BertForPreTraining.from_pretrained(BASE_BERT_MODEL)

bert_model.resize_token_embeddings(len(temp_tokenizer))

bert_model.save_pretrained(CHAR_ADD_MODEL_PATH)

3154
{'噫': 7, 'に': 28190, 'つ': 6068, 'い': 19551, 'て': 16027, 'の': 38771, '説': 3689, '明': 4270, '。': 36093, 'は': 26836, '「': 9632, 'あ': 8303, '」': 9649, 'と': 22530, '読': 3538, 'む': 4084, '噯': 29, '気': 7635, 'こ': 7570, '〈': 1994, '『': 2043, '霊': 383, '枢': 435, '』': 2027, '口': 1423, '問': 548, '篇': 885, '〉': 1981, '☞': 924, '曖': 12, 'き': 4088, '傷': 1436, '寒': 2721, '論': 939, '弁': 273, '太': 931, '陽': 3881, '病': 5571, '脈': 4081, '証': 4196, '併': 244, '治': 4394, '体': 2545, 'が': 20426, '胃': 2167, '中': 2168, 'か': 4335, 'ら': 5306, '咽': 330, '喉': 526, '向': 290, 'っ': 4355, '上': 1913, '逆': 520, 'し': 15748, '、': 35482, '音': 108, 'を': 20072, '伴': 1246, 'う': 8364, '症': 3643, '状': 3529, '(': 2960, 'げ': 445, 'ぷ': 51, ')': 2973, '指': 2057, 'す': 18234, '丹': 250, '渓': 37, '心': 2636, '法': 4144, 'そ': 2018, '重': 686, 'く': 5535, '長': 697, '呃': 54, 'よ': 6902, 'な': 13386, '急': 561, '迫': 205, '短': 171, 'で': 12366, '【': 375, '参': 401, '照': 226, '】': 375, 'も': 4799, '現': 1563, '代': 648, '多': 1845, '場': 874, '合': 191

/home/itohiroki/kubert/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


30101
0
8771
{'「補」と[', '芩)', '臨床では、咳喘と食欲不振・納', 'an inflammatory infection of the testis and epididymis marked by local pain and swelling, referring to epididymitis and orchitis', 'greater yang disease pattern/syndrome」のカテゴリに属す', '肝氣犯脾は、「liver qi invading the spleen」のカテゴリに属す', 'pattern/syndrome of loss of smell due to qi deficiensyと同義', 'bluish discoloration of the bulbar conjunctiva surrounding the cornea after recurrent inflammation of the sclera with violet bulging', 'true excess with false deficiency」のカテゴリに属す', 'mind とも呼ばれる', 'a pathological change in which deficient liver-kidney yin lets liver yang get out of control and stir upward', 'a general term referring to various functional disorders of the stomach, i.e., dysfunction in receiving and digesting food as well as in conducting the contents to the intestines', 'throat wind」のカテゴリに属す', 'wind syncope」のカテゴリに属す', '呃)', 'effulgent heartliver fire」のカテゴリに属す', 'spiritual activities', 'insecurity of exterior qi」のカテゴリに属す', '「食亦」は、多食・すぐ空腹にな

Some weights of BertForPreTraining were not initialized from the model checkpoint at alabnii/jmedroberta-base-sentencepiece and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


## 定義済みの単語などを追加する

辞書などで定義された単語を追加した後、未知語になった個所を追加する操作

In [3]:
import json

PATH_DATASET_JSON = "../../resources/structured_data/chui_word_dic.json"
PATH_CORPUS = "../../resources/corpus/chui_word_dic_corpus.txt"
OUTPUT_TOKENIZER_PATH = "./tmbert_bf_randinit"

# 入れ子になった中医用語辞典を入れ子じゃなくする
def unnest_dict(dict_data)-> dict[str, dict]:
    data_list = {}
    for k0, v0 in dict_data.items():
        if v0["type"] == "content" and "content" in v0.keys():
            copy_k0 = k0.replace("―", "") if k0[0] == "―" or k0[-1] == "―" else k0
            data_list[copy_k0] = v0["content"]
        if "child" in v0.keys():
            for k1, v1 in v0["child"].items():
                if v1["type"] == "content" and "content" in v1.keys():
                    copy_k1 = k1.replace("―", "") if k1[0] == "―" or k1[-1] == "―" else k1
                    data_list[copy_k1] = v1["content"]
                if "child" in v1.keys():
                    for k2, v2 in v1["child"].items():
                        if v2["type"] == "content" and "content" in v2.keys(): 
                            copy_k2 = k2.replace("―", "") if k2[0] == "―" or k2[-1] == "―" else k2
                            data_list[copy_k2] = v2["content"]
    return data_list


import re
import jaconv
import json

# 除外するパターンを指定
exclude_patterns = []
exclude_patterns.append(r'(?a)\w+')  # 1文字以上のACSII文字で構成される文字列 (a-z, A-Z, 数字)
# exclude_patterns.append(r'[ぁ-ん].*$')  # ひらがなのみが含まれる文字列
# exclude_patterns.append(r'[ァ-ヶ]$')  # カタカナのみが含まれる文字列
exclude_patterns.append(r'^(?!\w).*$')  # 記号のみが含まれる文字列

# ひらがな
exclude_patterns.append("[\u3041-\u309F]+")

# カタカナ
exclude_patterns.append("[\u30A1-\u30FF]+")
exclude_patterns.append("[\uFF66-\uFF9F]+") # 半角
exclude_patterns.append("[①-㊿]+")

# ASCII記号の全角版および日本語の記号の半角版
exclude_patterns.append("[\uFF01-\uFF0F\uFF1A-\uFF20\uFF3B-\uFF40\uFF5B-\uFF65\u3000-\u303F]+")
compiled_patterns: list = [re.compile(p) for p in exclude_patterns]

def preprocess_text(text: str)-> str:
    text = jaconv.h2z(text) # 半角カタカナ→全角カタカナ
    text = jaconv.z2h(text, kana=False, digit=True) # 全角数字→半角数字
    return text

# 漢字だけ取り出してリストにする
def split_kanji_text(text: str)-> list:
    char_list: list = [char for char in text]
    word_list = []
    word_char = []
    for char in char_list:
        is_match = False
        for r in compiled_patterns:
            is_match |= r.fullmatch(char) is not None
        if is_match and len(word_char) != 0:
            word_list.append("".join(word_char))
            word_char.clear()
        elif not is_match:
            word_char.append(char)
    if len(word_char) != 0:
        word_list.append("".join(word_char))
    return word_list


with open(PATH_DATASET_JSON, mode="r", encoding="utf-8") as fp:
    dict_data = json.load(fp)


dict_data = unnest_dict(dict_data)


kanji_word_list = []
for sho_name, sentence_list in dict_data.items():
    for sentence in sentence_list:
        clean_text = preprocess_text(sentence)
        kanji_word_list.extend(split_kanji_text(clean_text))

kanji_word_list = sorted(list(set(kanji_word_list)), key=len)


from transformers import AutoTokenizer, AutoModel
import json

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(BASE_BERT_MODEL)

with open(PATH_DATASET_JSON, mode="r", encoding="utf-8") as fp:
    raw_dict_data = json.load(fp)


def split_multi_str(text: str, split_chars: list[str])-> list[str]:
    for split_char in split_chars:
        if split_char in text:
            return text.split(split_char)
    return [text]


def get_dict_words(raw_dict_data):
    word_list = []
    word_list.append("☞")
    for k0, v0 in raw_dict_data.items():
        if v0["type"] == "content":
            word_list.extend(split_multi_str(k0, ["/", "・", "、"]))
        if "child" in v0.keys():
            for k1, v1 in v0["child"].items():
                if v1["type"] == "content":
                    word_list.extend(split_multi_str(k1, ["/", "・", "、"]))
                if "child" in v1.keys():
                    for k2, v2 in v1["child"].items():
                        if v2["type"] == "content":
                            word_list.extend(split_multi_str(k2, ["/", "・", "、"]))
    return word_list


word_list = get_dict_words(raw_dict_data)

word_list = [word.replace("―", "") if word[0] == "―" or word[-1] == "―" else word for word in word_list]
word_list = list(set(word_list))
word_list.sort(key=len, reverse=True)
tokenizer.add_tokens(list(word_list))

tokenizer.save_pretrained("./temp_tokenizer")

import json
import unicodedata

# トークナイザですべての文章を処理して、[UNK]トークンに概要する語彙をリストに保存
unk_tokenize_word_set = set()

with open(PATH_CORPUS, mode="r", encoding="utf-8") as fp:
    for line in fp.readlines():
        if line == "\n":
            continue
        sentence = unicodedata.normalize('NFKC', line.strip("\n").strip())
        ids = tokenizer.encode(sentence, return_tensors='pt')
        tokens = tokenizer.convert_ids_to_tokens(ids[0])
        if ("[UNK]" in tokens):
            new_sentence = sentence
            for i in range(0, len(tokens)):
                tokens[i] = tokens[i].removeprefix("##")

            for i in range(0, len(tokens)):
                if (tokens[i] == "[UNK]"):
                    next_known_token = ""
                    for j in  range(i + 1, len(tokens)):
                        if (tokens[j] != "[UNK]"):
                            next_known_token = tokens[j]
                            break
                    if next_known_token != "":
                        next_token_start_index = new_sentence.find(next_known_token)
                        new_word = new_sentence[:next_token_start_index]
                        if new_word != "":
                            unk_tokenize_word_set.add(new_word)
                        new_sentence = new_sentence[next_token_start_index:]
                    else:
                        if new_sentence != "":
                            unk_tokenize_word_set.add(new_sentence)
                        break
                else:
                    #print(f"remove {wakatis[i]} before + " + new_sentence)
                    new_sentence = new_sentence.removeprefix(tokens[i])
                    # print("after  + " + new_sentence)

unk_token_list = list(unk_tokenize_word_set)

defined_kanji = []

from transformers import AutoTokenizer, AutoModel
import bisect

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(BASE_BERT_MODEL)
vocab_dic = tokenizer.get_vocab()
for k, v in vocab_dic.items():
    temp_list = split_kanji_text(k)
    if len(temp_list) == 1:
        defined_kanji.extend(temp_list)

defined_kanji = sorted(list(set(defined_kanji)), key=len)

kanji_word_list.sort(key=len)
add_word_list = []

for unk_token in unk_token_list:
    temp_list = []
    for kanji_word in kanji_word_list:
        if unk_token in kanji_word:
            temp_list.append(kanji_word)
    add_word_list.extend(temp_list)

add_word_list = list(set(add_word_list))
add_word_list.sort(key=len)

max_len = len(add_word_list[-1])
defined_word_dic = {}

for i in range(1, max_len + 1):
    defined_word_dic[i] = []

for kanji in defined_kanji:
    kanji_len = len(kanji)
    defined_word_dic[kanji_len].append(kanji)

new_word_list = []

for kanji in kanji_word_list:
    ids = tokenizer.encode(kanji, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(ids[0])
    if "[UNK]" in tokens:
        new_word_list.append(kanji)

new_word_list.sort(key=len)

from transformers import AutoTokenizer, AutoModel
import json

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(BASE_BERT_MODEL)

word_list = get_dict_words(raw_dict_data)

word_list = [word.replace("―", "") if word[0] == "―" or word[-1] == "―" else word for word in word_list]

kanji_word_list = new_word_list
kanji_word_list = [kanji for kanji in kanji_word_list if True]
word_list.extend(kanji_word_list)
word_list = list(set(word_list))
word_list.sort(key=len, reverse=True)
tokenizer.add_tokens(list(word_list))

tokenizer.save_pretrained(OUTPUT_TOKENIZER_PATH)


('./tmbert_bf_randinit/tokenizer_config.json',
 './tmbert_bf_randinit/special_tokens_map.json',
 './tmbert_bf_randinit/tokenizer.json')